# Tikhonov regularization with KL data fidelity term
We consider the two-dimensional deconvolution problems to find a non-negative function f given data 
$$
    d \sim \mathrm{Pois}(h*f)
$$
with a non-negative convolution kernel $h$, and $\mathrm{Pois}$ denotes the element-wise Poisson distribution.

We first study constrained quadratic Tikhonov regularization 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\mathrm{KL}(d+\sigma,h*f+\sigma) + \alpha \|f\|^2_{L^2}\right]
$$
with the Kullback-Leibler divergence as data fidelity term and an offset $\sigma>0$.
We also test entropy regularization given by 
$$
\hat{f} = \mathrm{argmin}_{f\geq 0} \left[\mathrm{KL}(d+\sigma,h*f+\sigma) + \alpha \mathrm{KL}(f,1)\right]
$$

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mplib

from regpy.operators.convolution import GaussianBlur
from regpy.vecsps import UniformGridFcts
from regpy.solvers import TikhonovRegularizationSetting, RegularizationSetting
from regpy.solvers.linear.semismoothNewton import SemismoothNewton_nonneg
from regpy.solvers.linear.proximal_gradient import ForwardBackwardSplitting, FISTA
from regpy.solvers.linear.primal_dual import PDHG
from regpy.hilbert import L2
from regpy.stoprules import DualityGapStopping
from regpy.functionals import QuadraticLowerBound, QuadraticBilateralConstraints, KullbackLeibler, RelativeEntropy

from comparison_plot import comparison_plot
from test_images import mixed

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)

### creating Poisson distributed synthetic data

In [ ]:
grid, exact_sol = mixed(M=256,N=256,fac=200.)
r"""grid is the underlying UniformGridFcts vector space, and exact_sol the exact solution."""
a=0.15
conv =  GaussianBlur(grid,a,pad_amount=16)
r"""Convolution operator $f\mapsto h*f$ for the convolution kernel $h(x)=\exp(-|x|_2^2/a^2)$."""
blur = conv(exact_sol)
blur[blur<0] = 0.
"""Simulated exact data."""
data = np.random.poisson(blur)
"""Simulated measured data. The Poisson distribution occurs if photon count detectors are used."""
comparison_plot(grid,exact_sol,data,title_left='noisy measurement data')

## Kullback-Leibler data fidelity with nonnegativity-contrained $L^2$ penalty

In [ ]:
sigma = 50.
KL_shift = KullbackLeibler(grid,w=data+sigma).shift(np.broadcast_to(-sigma,grid.shape))
KL_shift_lin = KullbackLeibler(grid,w=data+sigma, quad_taylor_l=sigma).shift(np.broadcast_to(-sigma,grid.shape))
penLower = QuadraticLowerBound(grid,x0=0,lb=0)
alpha = 1e-3
settingKL_Lower = TikhonovRegularizationSetting(op=conv, penalty=penLower, data_fid = KL_shift,regpar=alpha)
settingKL_lin_Lower = TikhonovRegularizationSetting(op=conv, penalty=penLower, data_fid = KL_shift_lin,regpar=alpha)

### Forward-backward splitting

In [ ]:
n_iter = 400 
FB_KL_lower = ForwardBackwardSplitting(settingKL_Lower)
stop_FB_KL_lower=DualityGapStopping(FB_KL_lower,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_=FB_KL_lower.run(stoprule=stop_FB_KL_lower)

FB_KL_lin_lower = ForwardBackwardSplitting(settingKL_lin_Lower)
stop_FB_KL_lin_lower=DualityGapStopping(FB_KL_lin_lower,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_= FB_KL_lin_lower.run(stoprule=stop_FB_KL_lin_lower)

#comparison_plot(grid,exact_sol,FB_KL_lower.x,title_left='Forward-backward')

### FISTA

In [ ]:
n_iter=1000
#FISTA_KL_lower = FISTA(settingKL_Lower,init = 100*grid.ones())
#stop_FISTA_KL_lower=DualityGapStopping(FISTA_KL_lower,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
#FISTA_KL_lower.run(stoprule=stop_FISTA_KL_lower)

FISTA_KL_lin_lower = FISTA(settingKL_lin_Lower,init = 100*grid.ones())
stop_FISTA_KL_lin_lower=DualityGapStopping(FISTA_KL_lin_lower,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_=FISTA_KL_lin_lower.run(stoprule=stop_FISTA_KL_lin_lower)

#comparison_plot(grid,exact_sol,FISTA_KL_lower.x,title_left='FISTA reco')

### Primal-dual hybrid gradient (PDHG) method

In [ ]:
PDHG_KL_lower = PDHG(settingKL_Lower)
stop_PDHG_KL_lower = DualityGapStopping(PDHG_KL_lower,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_KL_lower.run(stoprule=stop_PDHG_KL_lower)

PDHG_KL_lin_lower = PDHG(settingKL_lin_Lower)
stop_PDHG_KL_lin_lower = DualityGapStopping(PDHG_KL_lin_lower,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_=PDHG_KL_lin_lower.run(stoprule=stop_PDHG_KL_lin_lower)

#comparison_plot(grid,exact_sol,PDHG_KL_lower.x, title_left='PDHG reco')

#comparison_plot(grid,exact_sol,PDHG_KL_lower.x, title_left='PDHG reco')

### comparison of convergence speeds

In [ ]:
plt.semilogy(stop_FB_KL_lower.gap_stat,'b-',label='ForwardBackward')
plt.semilogy(stop_FB_KL_lin_lower.gap_stat,'b:',label='ForwardBackward')
#plt.semilogy(stop_FISTA_KL_lower.gap_stat,'r.',label='FISTA')
plt.semilogy(stop_FISTA_KL_lin_lower.gap_stat,'r:',label='FISTA')
plt.semilogy(stop_PDHG_KL_lower.gap_stat,'g-',label='PDHG')
plt.semilogy(stop_PDHG_KL_lin_lower.gap_stat,'g:',label='PDHG')
plt.legend()
plt.xlabel('it. step'); plt.ylabel('duality gap')
plt.title('convergence for nonnegativity constraint')

## Kullback-Leibler data fidelity with relative entropy penalty

In [ ]:
RE = RelativeEntropy(grid,w = 300*grid.ones())
RE_ub = RelativeEntropy(grid,w = 300*grid.ones(),constr_u=400)
alpha = 1e-5/RE_ub.convexity_param
setting_KL_RE = TikhonovRegularizationSetting(op=conv, penalty=RE, data_fid = KL_shift_lin,regpar=alpha)
setting_KL_RE_ub = TikhonovRegularizationSetting(op=conv, penalty=RE_ub, data_fid = KL_shift_lin,regpar=alpha)

### Forward-backward splitting

In [ ]:
n_iter = 400
FB_KL_RE = ForwardBackwardSplitting(setting_KL_RE,init= RE.w)#10*grid.ones())
stop_FB_KL_RE=DualityGapStopping(FB_KL_RE,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
#_ = FB_KL_RE.run(stoprule=stop_FB_KL_RE)

FB_KL_RE_ub = ForwardBackwardSplitting(setting_KL_RE_ub,init=RE.w)
stop_FB_KL_RE_ub=DualityGapStopping(FB_KL_RE_ub,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_ = FB_KL_RE_ub.run(stoprule=stop_FB_KL_RE_ub)

comparison_plot(grid,exact_sol,FB_KL_RE.x,title_left='Forward-backward rel. entropy')

### FISTA

In [ ]:
FISTA_KL_RE = FISTA(setting_KL_RE,init = RE.w)
stop_FISTA_KL_RE=DualityGapStopping(FISTA_KL_RE,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
#_=FISTA_KL_RE.run(stoprule=stop_FISTA_KL_RE)

FISTA_KL_RE_ub = FISTA(setting_KL_RE_ub, init =RE.w)
stop_FISTA_KL_RE_ub=DualityGapStopping(FISTA_KL_RE_ub,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_=FISTA_KL_RE_ub.run(stoprule=stop_FISTA_KL_RE_ub)

comparison_plot(grid,exact_sol,FISTA_KL_RE_ub.x,title_left='FISTA rel. entropy')


### Primal-dual hybrid gradient (PDHG) method

In [ ]:
PDHG_KL_RE = PDHG(setting_KL_RE)#,init_domain=np.ones_like(RE.w))
stop_PDHG_KL_RE=DualityGapStopping(PDHG_KL_RE,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_KL_RE.run(stoprule=stop_PDHG_KL_RE)

PDHG_KL_RE_ub = PDHG(setting_KL_RE_ub)#,init_domain=np.ones_like(RE.w))
stop_PDHG_KL_RE_ub = DualityGapStopping(PDHG_KL_RE_ub,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
PDHG_KL_RE_ub.run(stoprule=stop_PDHG_KL_RE_ub)

comparison_plot(grid,exact_sol,PDHG_KL_RE.x, title_left='PDHG reco')

### comparison of convergence speeds

In [ ]:
plt.semilogy(stop_FB_KL_RE.gap_stat,'b-',label='ForwardBackward')
plt.semilogy(stop_FISTA_KL_RE_ub.gap_stat,'r-',label='FISTA')
plt.semilogy(stop_PDHG_KL_RE.gap_stat,'g-',label='PDHG')
plt.semilogy(stop_PDHG_KL_RE_ub.gap_stat,'g:',label='PDHG, RE with ub')
plt.legend()
plt.xlabel('it. step'); plt.ylabel('duality gap')
plt.title('convergence for relative entropy regularization')

In [ ]:
setting_KLstd_RE_ub = TikhonovRegularizationSetting(op=conv, penalty=RE, data_fid = KL_shift,regpar=alpha)
dual_KLstd_RE_ub = setting_KLstd_RE_ub.dualSetting()
dual_FISTA_KLstd_RE_ub = FISTA(dual_KLstd_RE_ub)
stop_dual_FISTA_KLstd_RE=DualityGapStopping(dual_FISTA_KLstd_RE_ub,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_= dual_FISTA_KLstd_RE_ub.run(stoprule=stop_dual_FISTA_KLstd_RE)

In [ ]:
dual_KL_RE_ub = setting_KL_RE_ub.dualSetting()
dual_FISTA_KL_RE_ub = FISTA(dual_KL_RE_ub)
stop_dual_FISTA_KL_RE_ub=DualityGapStopping(dual_FISTA_KL_RE_ub,threshold = 0.01, max_iter=n_iter,logging_level=logging.INFO)
pFISTA_dual, Tstar_pFISTA_dual= dual_FISTA_KL_RE_ub.run(stoprule=stop_dual_FISTA_KL_RE_ub)

f_FISTA_dual = dual_KL_RE_ub.dualToPrimal(Tstar_pFISTA_dual,argumentIsOperatorImage=True)

comparison_plot(grid,exact_sol,dual_FISTA_KL_RE_ub.x, title_left='dual FISTA reco')

In [ ]:
dual_PDHG_KL_RE_ub = PDHG(dual_KL_RE_ub)
stop_dual_PDHG_KL_RE_ub=DualityGapStopping(dual_PDHG_KL_RE_ub,threshold = 1., max_iter=n_iter,logging_level=logging.INFO)
_= dual_PDHG_KL_RE_ub.run(stoprule=stop_dual_PDHG_KL_RE_ub)